In [1]:
# =========================================================
# PS S6E4 - exp_20260410_035_dao_xgb_pair_te_plus_formula_min
# 034 core-min + Traiko formula/logit graft
# =========================================================

## Import / config

In [2]:
import os
import gc
import json
import random
from itertools import combinations

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight


class CFG:
    EXP_ID = "exp_20260410_035_dao_xgb_pair_te_plus_formula_min"
    SEED = 42
    N_SPLITS = 5
    INNER_TE_CV = 5

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    PAIR_UNIQUE_RATIO_MAX = 0.5

    XGB_PARAMS = {
        "n_estimators": 20000,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 2.333941903991847,
        "subsample": 0.9766412297733108,
        "colsample_bytree": 0.535324419516146,
        "gamma": 4.258489082295074,
        "reg_alpha": 4.082875850185249e-08,
        "reg_lambda": 0.00013528868091784412,
        "objective": "multi:softprob",
        "num_class": 3,
        "tree_method": "hist",
        "enable_categorical": False,
        "eval_metric": "mlogloss",
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0,
    }

    # Traiko / Chris-side threshold features
    SOIL_THRESH = 25
    RAIN_THRESH = 300
    TEMP_THRESH = 30
    WIND_THRESH = 10

    # precomputed logits from Traiko notebook
    LOGIT_COEFS = {
        "Low": {
            "intercept": 16.3173,
            "soil_lt_25": -11.0237,
            "temp_gt_30": -5.8559,
            "rain_lt_300": -10.8500,
            "wind_gt_10": -5.8284,
            "Flowering": -5.4155,
            "Harvest": 5.5073,
            "Sowing": 5.2299,
            "Vegetative": -5.4617,
            "Mulch_No": -3.0014,
            "Mulch_Yes": 2.8613,
        },
        "Medium": {
            "intercept": 4.6524,
            "soil_lt_25": 0.3290,
            "temp_gt_30": -0.0204,
            "rain_lt_300": 0.1542,
            "wind_gt_10": 0.0841,
            "Flowering": 0.3586,
            "Harvest": -0.1348,
            "Sowing": -0.3547,
            "Vegetative": 0.3334,
            "Mulch_No": 0.1883,
            "Mulch_Yes": 0.0142,
        },
        "High": {
            "intercept": -20.9697,
            "soil_lt_25": 10.6947,
            "temp_gt_30": 5.8763,
            "rain_lt_300": 10.6958,
            "wind_gt_10": 5.7444,
            "Flowering": 5.0569,
            "Harvest": -5.3725,
            "Sowing": -4.8752,
            "Vegetative": 5.1283,
            "Mulch_No": 2.8131,
            "Mulch_Yes": -2.8755,
        },
    }


## Utils

In [3]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def build_pairwise_features_concat(train_df: pd.DataFrame,
                                   test_df: pd.DataFrame,
                                   all_cols: list[str],
                                   pair_unique_ratio_max: float = 0.5):
    n_train = len(train_df)

    train_pair_dict = {}
    test_pair_dict = {}
    pair_meta = []
    accepted_cols = []

    for c1, c2 in combinations(all_cols, 2):
        name = f"{c1}|{c2}"

        tr_s = train_df[c1].astype(str) + "_" + train_df[c2].astype(str)
        te_s = test_df[c1].astype(str) + "_" + test_df[c2].astype(str)

        combined = pd.concat([tr_s, te_s], axis=0, ignore_index=True)
        codes, uniques = pd.factorize(combined)

        nunique = pd.Series(codes).nunique()
        total_n = len(codes)

        if nunique > total_n * pair_unique_ratio_max:
            continue

        train_pair_dict[name] = codes[:n_train].astype("int32")
        test_pair_dict[name] = codes[n_train:].astype("int32")
        accepted_cols.append(name)

        pair_meta.append({
            "pair_name": name,
            "col1": c1,
            "col2": c2,
            "nunique_total": int(nunique),
            "n_rows_total": int(total_n),
            "ratio_unique": float(nunique / total_n),
        })

    train_pair_df = pd.DataFrame(train_pair_dict, index=train_df.index)
    test_pair_df = pd.DataFrame(test_pair_dict, index=test_df.index)
    pair_meta_df = pd.DataFrame(pair_meta)

    return train_pair_df, test_pair_df, accepted_cols, pair_meta_df


def apply_pair_te(X_tr_pair, y_tr, X_va_pair, X_te_pair, cols):
    enc = TargetEncoder(
        target_type="multiclass",
        cv=CFG.INNER_TE_CV,
        random_state=CFG.SEED
    )

    tr_enc = pd.DataFrame(enc.fit_transform(X_tr_pair[cols], y_tr), index=X_tr_pair.index)
    va_enc = pd.DataFrame(enc.transform(X_va_pair[cols]), index=X_va_pair.index)
    te_enc = pd.DataFrame(enc.transform(X_te_pair[cols]), index=X_te_pair.index)

    tr_enc.columns = [f"TE_{col}_cls{k}" for col in cols for k in range(3)]
    va_enc.columns = tr_enc.columns
    te_enc.columns = tr_enc.columns

    return tr_enc.reset_index(drop=True), va_enc.reset_index(drop=True), te_enc.reset_index(drop=True)


def add_formula_logit_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["soil_lt_25"] = (out["Soil_Moisture"] < CFG.SOIL_THRESH).astype("int8")
    out["temp_gt_30"] = (out["Temperature_C"] > CFG.TEMP_THRESH).astype("int8")
    out["rain_lt_300"] = (out["Rainfall_mm"] < CFG.RAIN_THRESH).astype("int8")
    out["wind_gt_10"] = (out["Wind_Speed_kmh"] > CFG.WIND_THRESH).astype("int8")

    stage_flowering = (out["Crop_Growth_Stage"] == "Flowering").astype("int8")
    stage_harvest = (out["Crop_Growth_Stage"] == "Harvest").astype("int8")
    stage_sowing = (out["Crop_Growth_Stage"] == "Sowing").astype("int8")
    stage_vegetative = (out["Crop_Growth_Stage"] == "Vegetative").astype("int8")

    mulch_no = (out["Mulching_Used"] == "No").astype("int8")
    mulch_yes = (out["Mulching_Used"] == "Yes").astype("int8")

    for cls_name in ["Low", "Medium", "High"]:
        coef = CFG.LOGIT_COEFS[cls_name]

        out[f"logit(P(y={cls_name}))"] = (
            coef["intercept"]
            + coef["soil_lt_25"] * out["soil_lt_25"]
            + coef["temp_gt_30"] * out["temp_gt_30"]
            + coef["rain_lt_300"] * out["rain_lt_300"]
            + coef["wind_gt_10"] * out["wind_gt_10"]
            + coef["Flowering"] * stage_flowering
            + coef["Harvest"] * stage_harvest
            + coef["Sowing"] * stage_sowing
            + coef["Vegetative"] * stage_vegetative
            + coef["Mulch_No"] * mulch_no
            + coef["Mulch_Yes"] * mulch_yes
        ).astype("float32")

    return out


seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

## Load data & FE

In [4]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)

num_features = [
    "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
    "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
    "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
]

cat_features = [
    "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
    "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
]

all_base_cols = num_features + cat_features

train_eng = train[[CFG.ID_COL] + all_base_cols + [CFG.TARGET]].copy()
test_eng = test[[CFG.ID_COL] + all_base_cols].copy()

# formula/logit graft
train_eng = add_formula_logit_features(train_eng)
test_eng = add_formula_logit_features(test_eng)

formula_features = [
    "soil_lt_25",
    "temp_gt_30",
    "rain_lt_300",
    "wind_gt_10",
    "logit(P(y=Low))",
    "logit(P(y=Medium))",
    "logit(P(y=High))",
]

# pairwise from original 19 raw columns only
train_pair_df, test_pair_df, te_columns, pair_meta_df = build_pairwise_features_concat(
    train_eng[all_base_cols],
    test_eng[all_base_cols],
    all_base_cols,
    pair_unique_ratio_max=CFG.PAIR_UNIQUE_RATIO_MAX
)

pair_meta_df.to_csv(f"{CFG.OUTPUT_DIR}/pair_feature_meta.csv", index=False)

# base now = raw 19 + formula 7
base_features = all_base_cols + formula_features

X_base = train_eng[base_features].copy()
X_tbase = test_eng[base_features].copy()
X_pair = train_pair_df.copy()
X_tpair = test_pair_df.copy()
y_full = train_eng[CFG.TARGET].values

# joint factorization for raw categorical base columns
for c in cat_features:
    combined = pd.concat([X_base[c].astype(str), X_tbase[c].astype(str)], axis=0, ignore_index=True)
    codes, uniques = pd.factorize(combined)
    X_base[c] = codes[:len(X_base)].astype("int32")
    X_tbase[c] = codes[len(X_base):].astype("int32")

# cast all base columns numeric
for c in X_base.columns:
    if X_base[c].dtype == "O":
        med = 0.0
        X_base[c] = X_base[c].astype("float32")
        X_tbase[c] = X_tbase[c].astype("float32")
    else:
        med = X_base[c].median()
        X_base[c] = X_base[c].fillna(med).astype("float32")
        X_tbase[c] = X_tbase[c].fillna(med).astype("float32")


## CV train

In [5]:
# ---------------------------------------------------------
# CV training
# ---------------------------------------------------------
skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED
)

oof_probs = np.zeros((len(X_base), 3), dtype=np.float32)
test_probs = np.zeros((len(X_tbase), 3), dtype=np.float32)

fold_scores = []
best_iterations = []
feature_importance_list = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_base, y_full), 1):
    print(f"\n===== Fold {fold}/{CFG.N_SPLITS} =====")

    X_tr_base = X_base.iloc[tr_idx].reset_index(drop=True)
    X_va_base = X_base.iloc[va_idx].reset_index(drop=True)
    X_te_base = X_tbase.reset_index(drop=True)

    y_tr = y_full[tr_idx]
    y_va = y_full[va_idx]

    X_tr_pair = X_pair.iloc[tr_idx].reset_index(drop=True)
    X_va_pair = X_pair.iloc[va_idx].reset_index(drop=True)
    X_te_pair = X_tpair.reset_index(drop=True)

    tr_te, va_te, te_te = apply_pair_te(
        X_tr_pair, y_tr, X_va_pair, X_te_pair, te_columns
    )

    Xt = pd.concat([X_tr_base, tr_te], axis=1).astype("float32")
    Xv = pd.concat([X_va_base, va_te], axis=1).astype("float32")
    Xe = pd.concat([X_te_base, te_te], axis=1).astype("float32")

    sw_tr = compute_sample_weight("balanced", y_tr)

    model = xgb.XGBClassifier(
        **CFG.XGB_PARAMS,
        early_stopping_rounds=300
    )

    model.fit(
        Xt,
        y_tr,
        sample_weight=sw_tr,
        eval_set=[(Xv, y_va)],
        verbose=500
    )

    oof_probs[va_idx] = model.predict_proba(Xv)
    test_probs += model.predict_proba(Xe) / CFG.N_SPLITS

    fold_score = balanced_accuracy_score(y_va, np.argmax(oof_probs[va_idx], axis=1))
    fold_scores.append(float(fold_score))
    best_iterations.append(int(getattr(model, "best_iteration", model.n_estimators)))

    booster_score = model.get_booster().get_score(importance_type="gain")
    feature_importance_list.append(booster_score)

    print(f"Fold {fold} BA: {fold_score:.6f}")

    del Xt, Xv, Xe, tr_te, va_te, te_te, model
    gc.collect()

cv_score = float(balanced_accuracy_score(y_full, np.argmax(oof_probs, axis=1)))
print(f"\nOOF Balanced Accuracy: {cv_score:.6f}")


===== Fold 1/5 =====
[0]	validation_0-mlogloss:1.05723
[500]	validation_0-mlogloss:0.05614
[1000]	validation_0-mlogloss:0.05191
[1500]	validation_0-mlogloss:0.05093
[2000]	validation_0-mlogloss:0.05076
[2500]	validation_0-mlogloss:0.05069
[3000]	validation_0-mlogloss:0.05064
[3500]	validation_0-mlogloss:0.05062
[4000]	validation_0-mlogloss:0.05060
[4500]	validation_0-mlogloss:0.05058
[5000]	validation_0-mlogloss:0.05057
[5500]	validation_0-mlogloss:0.05055
[6000]	validation_0-mlogloss:0.05054
[6500]	validation_0-mlogloss:0.05052
[7000]	validation_0-mlogloss:0.05051
[7500]	validation_0-mlogloss:0.05049
[8000]	validation_0-mlogloss:0.05048
[8299]	validation_0-mlogloss:0.05049
Fold 1 BA: 0.976996

===== Fold 2/5 =====
[0]	validation_0-mlogloss:1.05725
[500]	validation_0-mlogloss:0.05732
[1000]	validation_0-mlogloss:0.05317
[1500]	validation_0-mlogloss:0.05201
[2000]	validation_0-mlogloss:0.05185
[2500]	validation_0-mlogloss:0.05180
[3000]	validation_0-mlogloss:0.05174
[3500]	validation_0

## Save outputs

In [6]:
# ---------------------------------------------------------
# Save outputs
# ---------------------------------------------------------
np.save(f"{CFG.OUTPUT_DIR}/oof_dao_xgb_pair_te_plus_formula_min_proba.npy", oof_probs)
np.save(f"{CFG.OUTPUT_DIR}/pred_dao_xgb_pair_te_plus_formula_min_proba.npy", test_probs)

submission = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET: pd.Series(np.argmax(test_probs, axis=1)).map(CFG.INV_TARGET_MAP)
})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_dao_xgb_pair_te_plus_formula_min.csv", index=False)

feature_columns = pd.DataFrame({
    "feature_name": list(X_base.columns) + [f"TE_{col}_cls{k}" for col in te_columns for k in range(3)],
    "feature_group": (
        ["base_raw_or_formula"] * len(X_base.columns)
        + ["pair_te"] * (len(te_columns) * 3)
    )
})
feature_columns.to_csv(f"{CFG.OUTPUT_DIR}/feature_columns.csv", index=False)

all_feats = set()
for d in feature_importance_list:
    all_feats.update(d.keys())

rows = []
for feat in sorted(all_feats):
    vals = [imp.get(feat, 0.0) for imp in feature_importance_list]
    rows.append({
        "feature_name": feat,
        "importance_gain_mean": float(np.mean(vals)),
        "importance_gain_std": float(np.std(vals)),
    })

feature_importance_df = pd.DataFrame(rows).sort_values(
    "importance_gain_mean", ascending=False
)
feature_importance_df.to_csv(f"{CFG.OUTPUT_DIR}/feature_importance.csv", index=False)

# cv json
cv_result = {
    "exp_id": CFG.EXP_ID,
    "model": "XGBoost",
    "metric": "balanced_accuracy_score",
    "seed": CFG.SEED,
    "fold_type": "StratifiedKFold",
    "n_splits": CFG.N_SPLITS,
    "inner_te_cv": CFG.INNER_TE_CV,
    "base_feature_count": len(base_features),
    "formula_feature_count": len(formula_features),
    "pair_candidate_count": len(list(combinations(all_base_cols, 2))),
    "pair_accepted_count": len(te_columns),
    "fold_scores": fold_scores,
    "cv_score_raw": cv_score,
    "best_iterations": best_iterations,
    "xgb_params": CFG.XGB_PARAMS,
    "notes": {
        "pseudo_label": False,
        "posthoc_optimization": False,
        "formula_logit": True,
        "digit_features": False,
        "ordered_te": False,
        "pair_te_only": True
    }
}
save_json(cv_result, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("\nSaved files:")
for fn in [
    "oof_dao_xgb_pair_te_plus_formula_min_proba.npy",
    "pred_dao_xgb_pair_te_plus_formula_min_proba.npy",
    "submission_dao_xgb_pair_te_plus_formula_min.csv",
    "feature_columns.csv",
    "feature_importance.csv",
    "pair_feature_meta.csv",
    "cv_result.json",
]:
    print("-", f"{CFG.OUTPUT_DIR}/{fn}")


Saved files:
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/oof_dao_xgb_pair_te_plus_formula_min_proba.npy
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/pred_dao_xgb_pair_te_plus_formula_min_proba.npy
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/submission_dao_xgb_pair_te_plus_formula_min.csv
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/feature_columns.csv
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/feature_importance.csv
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/pair_feature_meta.csv
- /kaggle/working/exp_20260410_035_dao_xgb_pair_te_plus_formula_min/cv_result.json
